# W1 · Day 2 (v2) — Three-Seed Robustness with the Stabilized Trainer

Adds seeds 1 and 2 for OAI and NHANES using the **stabilized** within-one trainer
(gentle capped distance weighting, class-balance penalty, divergence detector,
all-grades checkpoint selection). Seed 0 already exists from the v2 run and is
skipped. This provides a three-seed mean so the within-one result is not a single
run (Guard 6). Writes only to scope3_w1.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, os
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import numpy as np, pandas as pd, json, glob, time, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
device='cuda' if torch.cuda.is_available() else 'cpu'
if device!='cuda': raise RuntimeError('No GPU.')
PROJECT_ROOT=Path('/content/drive/MyDrive/Master Thesis')
W1_ROOT=PROJECT_ROOT/'scope3_w1'; W1_CKPT=W1_ROOT/'checkpoints'; W1_RES=W1_ROOT/'results'
for d in (W1_ROOT,W1_CKPT,W1_RES): d.mkdir(parents=True, exist_ok=True)
manifest=TM.prepare_local_data()
def derive_patient_id(row):
    fn=str(row['filename']); stem=os.path.splitext(os.path.basename(fn))[0]
    for sep in ['_','-','.']:
        if sep in stem: stem=stem.split(sep)[0]; break
    return f"{row['dataset']}::{stem}"
manifest=manifest.copy(); manifest['patient_id']=manifest.apply(derive_patient_id,axis=1)
def within1(yt,yp): return float((np.abs(np.asarray(yt)-np.asarray(yp))<=1).mean())
def prediction_health(yp,n=5):
    c=np.bincount(np.asarray(yp),minlength=n); return int((c>0).sum()), c/c.sum()
print('Day-2 (v2) setup ready.')

Mounted at /content/drive
Copied array in 40s
Loaded array (61558, 224, 224) in 1s
Day-2 (v2) setup ready.


## Stabilized trainer (same as v2)

The identical stabilized loss and guarded loop from the v2 notebook, redefined here
so this notebook is self-contained. Runs skip any seed whose JSON already exists.

In [2]:
def assert_no_leak(tr,te,ds):
    assert ds not in set(tr['dataset'].unique()), f"LEAK: {ds} in training!"
    ov=set(tr['patient_id'])&set(te['patient_id']); assert len(ov)==0, f"LEAK: {len(ov)} shared ids!"

def stable_within1_loss(logits,y,lw,soft_alpha,dist_scale=0.15,dist_cap=1.6,epoch_frac=1.0):
    K=logits.shape[1]+1; tgt,mask=TM.corn_targets(y,K)
    if soft_alpha>0: tgt=tgt*(1.0-soft_alpha)+0.5*soft_alpha
    bce=F.binary_cross_entropy_with_logits(logits,tgt,reduction='none'); bce=(bce*mask).sum(1)/mask.sum(1).clamp(min=1)
    with torch.no_grad():
        pred=TM.corn_predict(logits); dist=(pred-y).abs().float()
        raw=1.0+(dist_scale*epoch_frac)*dist; wt=torch.clamp(raw,max=dist_cap); wt=wt/wt.mean().clamp(min=1e-6)
    loss=(bce*lw*wt).mean()
    cls=TM.corn_probs(logits); mean_mass=cls.mean(0)
    return loss + 0.05*((1.0/K - mean_mass).clamp(min=0)).sum()

def train_v2(run_name,test_ds,seed,dist_scale=0.15,dist_cap=1.6,log_fn=print):
    TM.set_seed(seed); done=W1_RES/f'{run_name}.json'
    if done.exists(): log_fn(f'[{run_name}] exists - skip'); return json.load(open(str(done)))
    tr,va,te=TM.make_splits(manifest,test_ds,seed); assert_no_leak(tr,te,test_ds)
    quality=TM.load_quality_weights()
    model=TM.OrdinalNet(config.NUM_CLASSES,4,True).to(device); model.freeze_stages(TM.MAX_FREEZE_STAGES)
    hp,bp=model.param_groups()
    opt=torch.optim.AdamW([{'params':hp,'lr':TM.MAX_LR_HEAD},{'params':bp,'lr':TM.MAX_LR_BACKBONE}],weight_decay=TM.WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda'); bs=TM.MAX_BATCH_SIZE; accum=TM.MAX_GRAD_ACCUM
    ckpt=W1_CKPT/f'{run_name}.pt'; ckb=W1_CKPT/f'{run_name}_best.pt'; e0,best,hist=0,0.0,[]
    if ckpt.exists():
        try: e0,best,hist=TM.load_ckpt(str(ckpt),model,opt)
        except Exception: e0,best,hist=0,0.0,[]
    no_imp=0; prev=None; from torch.utils.data import DataLoader
    for epoch in range(e0,TM.MAX_EPOCHS):
        if epoch<TM.CURRICULUM['phase1_end']: clean,mw=True,0.0
        elif epoch<TM.CURRICULUM['phase2_end']:
            clean=False; fr=(epoch-TM.CURRICULUM['phase1_end'])/max(1,TM.CURRICULUM['phase2_end']-TM.CURRICULUM['phase1_end']); mw=fr*TM.CURRICULUM['mrkr_target_weight']
        else: clean,mw=False,TM.CURRICULUM['mrkr_target_weight']
        loader=DataLoader(TM.KneeDataset(tr,True,quality),batch_size=bs,sampler=TM.build_sampler(tr,mw,clean),num_workers=TM.NUM_WORKERS,pin_memory=True)
        sc=(epoch+1)/TM.MAX_WARMUP if epoch<TM.MAX_WARMUP else 0.5*(1+math.cos(math.pi*(epoch-TM.MAX_WARMUP)/max(1,TM.MAX_EPOCHS-TM.MAX_WARMUP)))
        opt.param_groups[0]['lr']=TM.MAX_LR_HEAD*sc; opt.param_groups[1]['lr']=TM.MAX_LR_BACKBONE*sc
        grl=0.5*(2.0/(1.0+math.exp(-10*epoch/max(1,TM.MAX_EPOCHS)))-1.0)
        epoch_frac=max(0.0,min(1.0,(epoch-TM.MAX_WARMUP)/max(1,TM.MAX_EPOCHS-TM.MAX_WARMUP)))
        model.train(); t0=time.time(); rl=0.0; nb=0; opt.zero_grad()
        for bi,(x,y,lw,dom) in enumerate(loader):
            x,y,lw,dom=x.to(device),y.to(device),lw.to(device),dom.to(device)
            with torch.amp.autocast('cuda'):
                lo,s1,s2,dl=model(x,grl_lambda=grl)
                loss=stable_within1_loss(lo,y,lw,TM.MAX_SOFT_ALPHA,dist_scale,dist_cap,epoch_frac)+TM.hier_aux_loss(s1,s2,y)+TM.domain_loss(dl,dom)
                loss=loss/accum
            scaler.scale(loss).backward()
            if (bi+1)%accum==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),0.5); scaler.step(opt); scaler.update(); opt.zero_grad()
            rl+=loss.item()*accum; nb+=1
        eploss=rl/max(1,nb)
        if prev is not None and eploss>2.5*prev and ckb.exists():
            log_fn(f'  [{run_name}] divergence; revert+stop');
            try: TM.load_ckpt(str(ckb),model,None)
            except Exception: pass
            break
        prev=eploss
        vp,vpr=TM.predict_tta(model,va,device,bs,use_tta=False); vm=TM.compute_metrics(va['kl_grade'].values,vp,vpr); vw1=within1(va['kl_grade'].values,vp); u,_=prediction_health(vp)
        hist.append({'epoch':epoch,'val_within1':vw1,'val_acc':vm['acc5'],'val_qwk':vm['qwk'],'val_grades':u})
        log_fn(f"  [{run_name}] ep{epoch+1} val={vm['acc5']:.3f} w1={vw1:.3f} qwk={vm['qwk']:.3f} grades={u}/5 ({time.time()-t0:.0f}s)")
        score=vw1 if u>=5 else vw1*0.5
        if score>best:
            best=score
            try: TM.save_ckpt(str(ckb),model,opt,epoch,best,hist)
            except Exception: pass
            no_imp=0
        else: no_imp+=1
        try: TM.save_ckpt(str(ckpt),model,opt,epoch+1,best,hist)
        except Exception: pass
        if no_imp>=TM.MAX_PATIENCE: log_fn(f'  [{run_name}] early stop'); break
    if ckb.exists():
        try: TM.load_ckpt(str(ckb),model,None)
        except Exception: pass
    tp,tpr=TM.predict_tta(model,te,device,bs,use_tta=True); tm=TM.compute_metrics(te['kl_grade'].values,tp,tpr); tw1=within1(te['kl_grade'].values,tp); used,frac=prediction_health(tp)
    res={'run_name':run_name,'test_ds':test_ds,'seed':seed,'external_within1':tw1,'external_acc5':tm['acc5'],
         'external_qwk':tm['qwk'],'grades_used':used,'pred_distribution':[round(float(x),3) for x in frac]}
    np.savez_compressed(str(W1_RES/f'{run_name}_preds.npz'),y_true=te['kl_grade'].values,y_pred=tp,probs=tpr)
    json.dump(res,open(str(done),'w'),indent=2)
    log_fn(f"  [{run_name}] DONE within1={tw1:.3f} exact={tm['acc5']:.3f} qwk={tm['qwk']:.3f} grades={used}/5")
    return res
print('Stabilized trainer ready (skips existing seeds).')

Stabilized trainer ready (skips existing seeds).


## Run seeds 1 and 2 (OAI + NHANES)

In [3]:
for ds in ['oai','nhanes3']:
    for seed in [1,2]:
        rn=f'w1v2_{ds}_seed{seed}'; print(f'=== {rn} ==='); train_v2(rn, ds, seed=seed)
print('\nDay-2 (v2) seeds complete.')

=== w1v2_oai_seed1 ===
[w1v2_oai_seed1] exists - skip
=== w1v2_oai_seed2 ===
[w1v2_oai_seed2] exists - skip
=== w1v2_nhanes3_seed1 ===
[w1v2_nhanes3_seed1] exists - skip
=== w1v2_nhanes3_seed2 ===
[w1v2_nhanes3_seed2] exists - skip

Day-2 (v2) seeds complete.


## Three-seed summary

In [4]:
print('Three-seed within-1 (v2 stabilized):')
print(f"{'fold':10s}{'within1 mean±sd':>20s}{'exact':>10s}{'qwk':>9s}{'grades':>9s}")
for ds in ['oai','nhanes3']:
    fs=sorted(glob.glob(str(W1_RES/f'w1v2_{ds}_seed*.json')))
    w1=[];ex=[];qw=[];gr=[]
    for f in fs:
        r=json.load(open(f)); w1.append(r['external_within1']); ex.append(r['external_acc5']); qw.append(r['external_qwk']); gr.append(r['grades_used'])
    if w1:
        w1=np.array(w1)
        print(f"{ds:10s}{w1.mean():>12.3f}±{w1.std():.3f}   {np.mean(ex):>8.3f}{np.mean(qw):>9.3f}   {min(gr)}-{max(gr)}/5  (n={len(fs)})")
print('\nNow run Day 3 to ensemble these seeds + TTA + calibration.')

Three-seed within-1 (v2 stabilized):
fold           within1 mean±sd     exact      qwk   grades
oai              0.818±0.005      0.531    0.578   4-5/5  (n=3)
nhanes3          0.755±0.039      0.525    0.497   5-5/5  (n=3)

Now run Day 3 to ensemble these seeds + TTA + calibration.
